In [4]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Start local Spark session using all CPU cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PoetrySparkTest") \
    .getOrCreate()

# Print active Spark version
print(f"Spark version: {spark.version}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/15 23:50:30 WARN Utils: Your hostname, sahil-IdeaPad-Gaming-3-15ACH6, resolves to a loopback address: 127.0.1.1; using 192.168.29.208 instead (on interface wlo1)
26/09/15 23:50:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 23:50:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


In [5]:
import pyspark.sql.functions as F

In [6]:
from pathlib import Path

DATA_DIR = Path("../data/raw/foundation_foods")

food_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(DATA_DIR / "food.csv"))

foundation_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(DATA_DIR / "foundation_food.csv"))

food_nutrient_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(DATA_DIR / "food_nutrient.csv"))

nutrient_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(DATA_DIR / "nutrient.csv"))

food_category_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(DATA_DIR / "food_category.csv"))

In [7]:
food_df.printSchema()
foundation_df.printSchema()
food_nutrient_df.printSchema()
nutrient_df.printSchema()
food_category_df.printSchema()

root
 |-- fdc_id: string (nullable = true)
 |-- data_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- food_category_id: string (nullable = true)
 |-- publication_date: string (nullable = true)

root
 |-- fdc_id: integer (nullable = true)
 |-- NDB_number: integer (nullable = true)
 |-- footnote: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- fdc_id: integer (nullable = true)
 |-- nutrient_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- data_points: integer (nullable = true)
 |-- derivation_id: integer (nullable = true)
 |-- min: double (nullable = true)
 |-- max: double (nullable = true)
 |-- median: double (nullable = true)
 |-- footnote: string (nullable = true)
 |-- min_year_acquired: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- unit_name: string (nullable = true)
 |-- nutrient_nbr: double (nullable = true)
 |-- rank: double (nullable = true)



In [8]:
for name, df in {
    "food": food_df,
    "foundation_food": foundation_df,
    "food_nutrient": food_nutrient_df,
    "nutrient": nutrient_df,
    "food_category": food_category_df,
}.items():
    print(f"{name}: {len(df.columns)} columns")

print("Foods:", food_df.count())
print("Food nutrients:", food_nutrient_df.count())
print("Nutrients:", nutrient_df.count())
print("Categories:", food_category_df.count())

food: 5 columns
foundation_food: 3 columns
food_nutrient: 11 columns
nutrient: 5 columns
food_category: 3 columns
Foods: 87991
Food nutrients: 170469
Nutrients: 477
Categories: 28


In [9]:
nutrient_df.filter(
    F.lower(F.col("name")).rlike(
        "energy|protein|carbohydrate|lipid|fat|fiber|sugar|sodium"
    )
).show(truncate=False)

[Stage 22:>                                                         (0 + 1) / 1]

+----+------------------------------------------------+---------+------------+--------+
|id  |name                                            |unit_name|nutrient_nbr|rank    |
+----+------------------------------------------------+---------+------------+--------+
|2047|Energy (Atwater General Factors)                |KCAL     |957.0       |280.0   |
|2048|Energy (Atwater Specific Factors)               |KCAL     |958.0       |290.0   |
|1003|Protein                                         |G        |203.0       |600.0   |
|1004|Total lipid (fat)                               |G        |204.0       |800.0   |
|1005|Carbohydrate, by difference                     |G        |205.0       |1110.0  |
|1006|Fiber, crude (DO NOT USE - Archived)            |G        |206.0       |999999.0|
|1008|Energy                                          |KCAL     |208.0       |300.0   |
|1049|Solids, non-fat                                 |G        |253.0       |999999.0|
|1050|Carbohydrate, by summation

In [10]:
food_df.select(
    "fdc_id",
    "data_type",
    "description",
    "food_category_id"
).show(10, truncate=False)

+------+------------------+---------------------+----------------+
|fdc_id|data_type         |description          |food_category_id|
+------+------------------+---------------------+----------------+
|319874|sample_food       |HUMMUS, SABRA CLASSIC|16              |
|319875|market_acquisition|HUMMUS, SABRA CLASSIC|16              |
|319876|market_acquisition|HUMMUS, SABRA CLASSIC|16              |
|319877|sub_sample_food   |Hummus               |16              |
|319878|sub_sample_food   |Hummus               |16              |
|319879|sample_food       |HUMMUS, SABRA CLASSIC|16              |
|319880|market_acquisition|HUMMUS, SABRA CLASSIC|16              |
|319881|market_acquisition|HUMMUS, SABRA CLASSIC|16              |
|319882|sub_sample_food   |Hummus               |16              |
|319883|sub_sample_food   |Hummus               |16              |
+------+------------------+---------------------+----------------+
only showing top 10 rows


In [11]:
food_df.groupBy("data_type").count().orderBy("count", ascending=False).show(truncate=False)

+------------------------+-----+
|data_type               |count|
+------------------------+-----+
|sub_sample_food         |75055|
|market_acquisition      |7577 |
|sample_food             |4079 |
|agricultural_acquisition|810  |
|foundation_food         |469  |
|2019-04-01              |1    |
+------------------------+-----+



In [12]:
foundation_foods = food_df.filter(
    F.col("data_type") == "foundation_food"
)

foundation_foods.show(5, truncate=False)
foundation_nutrients = food_nutrient_df.join(
    foundation_foods.select("fdc_id"),
    on="fdc_id",
    how="inner"
)

foundation_nutrients.count()

+------+---------------+------------------------------------------------------------------------+----------------+----------------+
|fdc_id|data_type      |description                                                             |food_category_id|publication_date|
+------+---------------+------------------------------------------------------------------------+----------------+----------------+
|321358|foundation_food|Hummus, commercial                                                      |16              |2019-04-01      |
|321359|foundation_food|Milk, reduced fat, fluid, 2% milkfat, with added vitamin A and vitamin D|1               |2019-04-01      |
|321360|foundation_food|Tomatoes, grape, raw                                                    |11              |2019-04-01      |
|321505|foundation_food|Salt, table, iodized                                                    |2               |2019-04-01      |
|321611|foundation_food|Beans, snap, green, canned, regular pack, drained so

21459

In [13]:
foundation_nutrients_with_names = foundation_nutrients.join(
    nutrient_df,
    foundation_nutrients.nutrient_id == nutrient_df.id,
    "inner"
)

foundation_nutrients_with_names.select(
    foundation_nutrients.fdc_id,
    "name",
    "unit_name",
    "amount"
).show(100, truncate=False)

+------+-----------------------------------+---------+------+
|fdc_id|name                               |unit_name|amount|
+------+-----------------------------------+---------+------+
|321358|SFA 10:0                           |G        |0.0   |
|321358|SFA 16:0                           |G        |1.41  |
|321358|PUFA 20:5 n-3 (EPA)                |G        |0.0   |
|321358|Tocopherol, delta                  |MG       |1.3   |
|321358|Magnesium, Mg                      |MG       |71.1  |
|321358|Galactose                          |G        |0.0   |
|321358|PUFA 20:2 n-6 c,c                  |G        |0.005 |
|321358|Fatty acids, total trans-dienoic   |G        |0.012 |
|321358|PUFA 18:3 n-3 c,c,c (ALA)          |G        |0.637 |
|321358|Choline, from glycerophosphocholine|MG       |1.1   |
|321358|SFA 4:0                            |G        |0.0   |
|321358|Choline, free                      |MG       |22.3  |
|321358|Lutein + zeaxanthin                |UG       |258.0 |
|321358|

In [14]:
nutrients_per_food = (
    foundation_nutrients
    .groupBy("fdc_id")
    .agg(F.countDistinct("nutrient_id").alias("nutrient_count"))
)

print("Foundation foods:", foundation_foods.count())
print("Total nutrient records:", foundation_nutrients.count())

nutrients_per_food.select(
    F.avg("nutrient_count").alias("average"),
    F.min("nutrient_count").alias("minimum"),
    F.max("nutrient_count").alias("maximum"),
    F.expr("percentile_approx(nutrient_count, 0.5)").alias("median")
).show()

Foundation foods: 469
Total nutrient records: 21459
+----------------+-------+-------+------+
|         average|minimum|maximum|median|
+----------------+-------+-------+------+
|45.8440170940171|     12|    159|    29|
+----------------+-------+-------+------+



In [15]:
nutrients_per_food.orderBy("nutrient_count").show(
    truncate=False
)

+-------+--------------+
|fdc_id |nutrient_count|
+-------+--------------+
|321505 |12            |
|789828 |13            |
|1104647|13            |
|746775 |13            |
|2758994|16            |
|2758996|16            |
|2758993|16            |
|2758998|16            |
|2758997|16            |
|2759000|16            |
|2758995|16            |
|2727574|18            |
|335912 |18            |
|2747670|18            |
|2727576|18            |
|335927 |18            |
|2727571|18            |
|747444 |18            |
|2747672|18            |
|335930 |18            |
+-------+--------------+
only showing top 20 rows


In [16]:
# Join with nutrient metadata
used_nutrients = (
    foundation_nutrients
    .join(
        nutrient_df,
        foundation_nutrients.nutrient_id == nutrient_df.id,
        "inner"
    )
    .groupBy(
        "nutrient_id",
        "name",
        "unit_name"
    )
    .agg(
        F.countDistinct("fdc_id").alias("foods_using_nutrient")
    )
    .orderBy(F.desc("foods_using_nutrient"))
)

used_nutrients.show(200, truncate=False)

+-----------+-------------------------------------------+---------+--------------------+
|nutrient_id|name                                       |unit_name|foods_using_nutrient|
+-----------+-------------------------------------------+---------+--------------------+
|1051       |Water                                      |G        |460                 |
|1089       |Iron, Fe                                   |MG       |439                 |
|1092       |Potassium, K                               |MG       |439                 |
|1091       |Phosphorus, P                              |MG       |439                 |
|1095       |Zinc, Zn                                   |MG       |439                 |
|1090       |Magnesium, Mg                              |MG       |439                 |
|1101       |Manganese, Mn                              |MG       |439                 |
|1098       |Copper, Cu                                 |MG       |439                 |
|1087       |Calcium,

In [17]:
nutrient_df.filter(
    F.lower(F.col("name")).rlike(
        "caffeine"
    )
).select(
    "id",
    "name",
    "unit_name"
).orderBy("name").show(
    100,
    truncate=False
)

+----+--------+---------+
|id  |name    |unit_name|
+----+--------+---------+
|1057|Caffeine|MG       |
+----+--------+---------+



In [18]:
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [19]:
from ingest.filters import filter_foundation_foods
from ingest.nutrients import select_application_nutrients
from ingest.transform import (
    join_foods_with_nutrients,
    pivot_nutrients,
)

foundation = filter_foundation_foods(food_df)

application_nutrients = select_application_nutrients(
    nutrient_df
)

joined = join_foods_with_nutrients(
    foundation,
    food_nutrient_df,
    application_nutrients,
)

foods = pivot_nutrients(joined)

(
    foods
    .filter(foods.carbohydrates < 0)
    .select(
        "fdc_id",
        "name",
        "carbohydrates",
    )
    .show(
        50,
        truncate=False,
    )
)

26/09/15 23:50:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+-------------------------------------------+-------------+
|fdc_id |name                                       |carbohydrates|
+-------+-------------------------------------------+-------------+
|2727568|Chicken, wing, meat and skin, raw          |-0.45875     |
|2747656|Halibut, frozen, wild caught               |-0.0602      |
|2727571|Bison, ground, raw                         |-0.14955     |
|2727569|Chicken, breast, meat and skin, raw        |-0.42825     |
|2747673|Tuna, ahi or yellowfin, frozen, wild caught|-0.1045      |
|2727567|Chicken, thigh, meat and skin, raw         |-0.17255     |
|2727566|Chicken, drumstick, meat and skin, raw     |-0.47505     |
|2727576|Pork, belly, with skin, raw                |-0.705       |
|2727575|Pork, chop, center cut, raw                |-0.5625      |
|2727570|Lamb, ground, raw                          |-0.2508      |
+-------+-------------------------------------------+-------------+

